# Homework 4
-   **Name:**  Victor Hugo Gomez Soto 
-  **e-mail:** -- victor.gomez2701@alumnos.udg.mx --


# MODULES

### Module Imports / Importación de Módulos
This section imports necessary libraries for numerical analysis, data manipulation, and visualization.

En esta sección se importan las bibliotecas necesarias para el análisis numérico, la manipulación de datos y la visualización.

### Importación de módulos
Se importan bibliotecas necesarias para el análisis numérico, la manipulación de datos y la visualización.

In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.spatial import distance
from scipy.stats import wrapcauchy, levy_stable
import math
import dash
from dash import dcc, html
from dash.dependencies import Input, Output



### Definition of Classes and Helper Functions / Definición de Clases y Funciones Auxiliares
This section defines classes and utility functions to facilitate trajectory computations and distance calculations in random motion models.

En esta sección se definen clases y funciones auxiliares para facilitar los cálculos de trayectorias y distancias en modelos de movimiento aleatorio.

# Functions

### Definición de Clases y Funciones Auxiliares
Se definen clases y funciones que facilitan el cálculo de trayectorias y distancias en los modelos de movimiento.

### Simulation of Random Walks / Simulación de Caminatas Aleatorias
Different random walk simulations are implemented, including Brownian Motion and Lévy Flights.

Se implementan distintas simulaciones de caminatas aleatorias, como el Movimiento Browniano y los Vuelos de Lévy.

In [4]:
# Nota: Esta clase la importaremos junto con el segundo bloque de modulos
################# http://www.pygame.org/wiki/2DVectorClass ##################
class Vec2d(object):
    """2d vector class, supports vector and scalar operators,
       and also provides a bunch of high level functions
       """
    __slots__ = ['x', 'y']

    def __init__(self, x_or_pair, y = None):
        if y == None:            
            self.x = x_or_pair[0]
            self.y = x_or_pair[1]
        else:
            self.x = x_or_pair
            self.y = y
            
    # Addition
    def __add__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x + other.x, self.y + other.y)
        elif hasattr(other, "__getitem__"):
            return Vec2d(self.x + other[0], self.y + other[1])
        else:
            return Vec2d(self.x + other, self.y + other)

    # Subtraction
    def __sub__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x - other.x, self.y - other.y)
        elif (hasattr(other, "__getitem__")):
            return Vec2d(self.x - other[0], self.y - other[1])
        else:
            return Vec2d(self.x - other, self.y - other)
    
    # Vector length
    def get_length(self):
        return math.sqrt(self.x**2 + self.y**2)
    
    # rotate vector
    def rotated(self, angle):        
        cos = math.cos(angle)
        sin = math.sin(angle)
        x = self.x*cos - self.y*sin
        y = self.x*sin + self.y*cos
        return Vec2d(x, y)
     # Método para convertir el vector en una tupla
    def to_tuple(self):
        return (self.x, self.y)

In [ ]:
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_2d(n_steps=1000, speed=5, start_pos=(0, 0)):    
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    
    for _ in range(n_steps):
        turn_angle = np.random.uniform(low=-np.pi, high=np.pi)
        step = Vec2d(speed, 0).rotated(turn_angle)
        pos += step
        trajectory.append(pos.to_tuple())
    
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_2d(n_steps=1000, speed=5, start_pos=(0, 0), c=0.5):   
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    
    angle = 0
    for _ in range(n_steps):
        delta_angle = wrapcauchy.rvs(c)
        angle += delta_angle
        step = Vec2d(speed, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())
    
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df

#####################################################################################
# Levi Flight  
#####################################################################################
def levy_flight(n_steps=1000, alpha=1.5, scale=1.0, c=0.5, start_pos=(0, 0)):
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    angle = 0  

    for _ in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))
        delta_angle = wrapcauchy.rvs(c)  
        angle += delta_angle
        step = Vec2d(step_size, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df

#####################################################################################
# path length 
#####################################################################################
def path_length(df):
    """Calcula la longitud total del camino recorrido."""
    distances = np.sqrt(np.diff(df["x_pos"])**2 + np.diff(df["y_pos"])**2)
    return np.sum(distances)


In [15]:
import pandas as pd
import numpy as np

def path_length(df):
    """Calcula la longitud total del camino recorrido."""
    distances = np.sqrt(np.diff(df["x_pos"])**2 + np.diff(df["y_pos"])**2)
    return np.sum(distances)

df_test = pd.DataFrame({
    "x_pos": np.random.randn(10).cumsum(),
    "y_pos": np.random.randn(10).cumsum()
})

print("Path Length:", path_length(df_test))

Path Length: 13.191611251627165


### Path Length Analysis / Análisis de la Longitud de Trayectoria


### Simulación de Caminatas Aleatorias
Aquí se implementan distintas simulaciones de caminatas aleatorias, como el movimiento Browniano y el vuelo de Lévy.

# Activity 1: Path length - BM1 vs BM2 vs CRW

### Mean Squared Displacement (MSD) Analysis / Análisis del Desplazamiento Cuadrático Medio (MSD)
The MSD is computed to analyze the relationship between time and displacement in each type of random walk.

Se calcula el MSD para analizar la relación entre el tiempo y la distancia recorrida en cada tipo de caminata aleatoria.

In [11]:
import pandas as pd

df = bm_2d(n_steps=500, speed=5, start_pos=(0, 0))
print(df.head())

      x_pos     y_pos
0  0.000000  0.000000
1  4.712545  1.670903
2  1.283701 -1.968194
3  5.575769  0.596598
4  1.490388  3.479246


In [ ]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import pandas as pd
import numpy as np
# Nota: Esta clase la importaremos junto con el segundo bloque de modulos
################# http://www.pygame.org/wiki/2DVectorClass ##################
class Vec2d(object):
    """2d vector class, supports vector and scalar operators,
       and also provides a bunch of high level functions
       """
    __slots__ = ['x', 'y']

    def __init__(self, x_or_pair, y = None):
        if y == None:            
            self.x = x_or_pair[0]
            self.y = x_or_pair[1]
        else:
            self.x = x_or_pair
            self.y = y
            
    # Addition
    def __add__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x + other.x, self.y + other.y)
        elif hasattr(other, "__getitem__"):
            return Vec2d(self.x + other[0], self.y + other[1])
        else:
            return Vec2d(self.x + other, self.y + other)

    # Subtraction
    def __sub__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x - other.x, self.y - other.y)
        elif (hasattr(other, "__getitem__")):
            return Vec2d(self.x - other[0], self.y - other[1])
        else:
            return Vec2d(self.x - other, self.y - other)
    
    # Vector length
    def get_length(self):
        return math.sqrt(self.x**2 + self.y**2)
    
    # rotate vector
    def rotated(self, angle):        
        cos = math.cos(angle)
        sin = math.sin(angle)
        x = self.x*cos - self.y*sin
        y = self.x*sin + self.y*cos
        return Vec2d(x, y)
     # Método para convertir el vector en una tupla
    def to_tuple(self):
        return (self.x, self.y)
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_2d(n_steps=1000, speed=5, start_pos=(0, 0)):    
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    
    for _ in range(n_steps):
        turn_angle = np.random.uniform(low=-np.pi, high=np.pi)
        step = Vec2d(speed, 0).rotated(turn_angle)
        pos += step
        trajectory.append(pos.to_tuple())
    
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_2d(n_steps=1000, speed=5, start_pos=(0, 0), c=0.5):   
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    
    angle = 0
    for _ in range(n_steps):
        delta_angle = wrapcauchy.rvs(c)
        angle += delta_angle
        step = Vec2d(speed, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())
    
    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    return df

#####################################################################################
# Levi Flight  
#####################################################################################
def levy_flight(n_steps=1000, alpha=1.5, scale=1.0, c=0.5, start_pos=(0, 0)):
    pos = Vec2d(*start_pos)
    trajectory = [pos.to_tuple()]
    angle = 0  

    for _ in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))
        delta_angle = wrapcauchy.rvs(c)  
        angle += delta_angle
        step = Vec2d(step_size, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())

    df = pd.DataFrame(trajectory, columns=["x_pos", "y_pos"])
    
    return df

#####################################################################################
# path length 
#####################################################################################
def path_length(trajectory):
    # Get the Euclidean Distance
    distances = np.array([distance.euclidean(trajectory.iloc[i-1], trajectory.iloc[i]) for i in range(1, trajectory.shape[0])])
    # Get the Cumulative Sum of the steps
    return np.cumsum(distances)
#####################################################################################
# turning angle distribution
#####################################################################################

def turning_angle_distribution(df):
    # Diferencias de posición en X e Y
    dx = df["x_pos"].diff().values[1:]  # Omitimos el primer valor NaN
    dy = df["y_pos"].diff().values[1:]

    # Construcción de los vectores de movimiento
    v1 = np.column_stack((dx[:-1], dy[:-1]))  # Primeros desplazamientos
    v2 = np.column_stack((dx[1:], dy[1:]))  # Segundos desplazamientos

    # Normalizamos los vectores
    norm_v1 = np.linalg.norm(v1, axis=1)
    norm_v2 = np.linalg.norm(v2, axis=1)

    # Evitar división por cero
    valid_indices = (norm_v1 > 0) & (norm_v2 > 0)
    v1, v2 = v1[valid_indices], v2[valid_indices]
    norm_v1, norm_v2 = norm_v1[valid_indices], norm_v2[valid_indices]

    # Producto punto y ángulos
    dot_product = np.einsum("ij,ij->i", v1, v2)
    cos_theta = dot_product / (norm_v1 * norm_v2)  # Cálculo del coseno
    cos_theta = np.clip(cos_theta, -1, 1)  # Evitar valores fuera de [-1,1] por errores numéricos

    angles = np.arccos(cos_theta)  # Convertir a ángulos en radianes
    return np.degrees(angles)  # Convertir a grados
#####################################################################################
# mean squared displacement
#####################################################################################
def mean_squared_displacement(df):

    x = df['x_pos'].values
    y = df['y_pos'].values
    N = len(x)
    msd = np.zeros(N)

    for t in range(N):
        dx = x[t:] - x[:N-t]  # Desplazamiento en X
        dy = y[t:] - y[:N-t]  # Desplazamiento en Y
        squared_displacement = dx**2 + dy**2  # MSD = dx² + dy²
        msd[t] = np.mean(squared_displacement)

    return msd

def turning_angle_distribution_old(df):
   
    v1 = np.stack((df["x_pos"].diff().iloc[1:-1], df["y_pos"].diff().iloc[1:-1]), axis=1)
    v2 = np.stack((df["x_pos"].diff().shift(-1).iloc[:-2], df["y_pos"].diff().shift(-1).iloc[:-2]), axis=1)

    # Verificar dimensiones
    if v1.shape != v2.shape:
        print(f"Error: dimensiones incompatibles -> v1: {v1.shape}, v2: {v2.shape}")
        return np.array([])

    # Producto punto
    dot_product = np.einsum("ij,ij->i", v1, v2)

    # Magnitud de los vectores
    norm_v1 = np.linalg.norm(v1, axis=1)
    norm_v2 = np.linalg.norm(v2, axis=1)

    # Evitar divisiones por cero
    cos_theta = np.clip(dot_product / (norm_v1 * norm_v2), -1.0, 1.0)

    # Calcular ángulos
    angles = np.arccos(cos_theta)
    
    return angles

# Inicializar la app Dash
app = dash.Dash(__name__)

# Definir función para generar una trayectoria
def generate_trajectory(traj_type='BM', n_steps=500, speed=5, alpha=1.5, scale=1.0, c=0.5):
    if traj_type == 'BM':
        df = bm_2d(n_steps, speed)
    elif traj_type == 'CRW':
        df = rw_2d(n_steps, speed, c=c)
    else:
        df = levy_flight(n_steps, alpha, scale, c)
    return df

# Estilos para mejorar la UI
styles = {
    'container': {
        'width': '80%',
        'margin': 'auto',
        'padding': '20px',
        'fontFamily': 'Arial, sans-serif',
        'backgroundColor': '#f8f9fa',
        'borderRadius': '10px',
        'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)'
    },
    'header': {
        'textAlign': 'center',
        'fontSize': '24px',
        'fontWeight': 'bold',
        'marginBottom': '20px'
    },
    'panel': {
        'padding': '15px',
        'border': '1px solid #ccc',
        'borderRadius': '5px',
        'backgroundColor': '#fff',
        'marginBottom': '15px'
    }
}

# Layout de la app
app.layout = html.Div(style=styles['container'], children=[
    html.H1("Simulación de Trayectorias Aleatorias", style=styles['header']),
    
    html.Div(style=styles['panel'], children=[
        html.Label("Selecciona el tipo de trayectoria:"),
        dcc.RadioItems(
            id='traj-selector',
            options=[
                {'label': 'Movimiento Browniano (BM)', 'value': 'BM'},
                {'label': 'Camino Aleatorio Correlacionado (CRW)', 'value': 'CRW'},
                {'label': 'Vuelo de Lévy (LF)', 'value': 'LF'}
            ],
            value='BM',
            inline=True
        )
    ]),

    html.Div(style=styles['panel'], children=[
        html.Label("Número de pasos:"),
        dcc.Slider(id='n-steps', min=100, max=1000, step=100, value=500, 
                   marks={i: str(i) for i in range(100, 1100, 200)})
    ]),

    html.Div(style=styles['panel'], children=[
        html.Label("Velocidad:"),
        dcc.Slider(id='speed', min=1, max=10, step=1, value=5)
    ]),

    html.Div(id='extra-params', style=styles['panel']),

    html.Div(style=styles['panel'], children=[
        html.Label("Selecciona la métrica a visualizar:"),
        dcc.Dropdown(
            id='metric-selector',
            options=[
                {'label': 'Path Length (PL)', 'value': 'PL'},
                {'label': 'Mean Squared Displacement (MSD)', 'value': 'MSD'},
                {'label': 'Turning Angle Distribution (TAD)', 'value': 'TAD'}
            ],
            value='PL'
        )
    ]),

    dcc.Graph(id='trajectory-plot'),
    dcc.Graph(id='metric-plot')
])

# Callback para mostrar parámetros adicionales
@app.callback(
    Output('extra-params', 'children'),
    [Input('traj-selector', 'value')]
)
def update_params(traj_type):
    if traj_type == 'BM':
        return ''
    return html.Div([
        html.Label("Coeficiente de Cauchy (CRW & LF):"),
        dcc.Slider(id='c-coefficient', min=0.1, max=1, step=0.1, value=0.5),
        html.Label("Exponente de Lévy (LF):"),
        dcc.Slider(id='alpha', min=0.5, max=2, step=0.1, value=1.5),
        html.Label("Escala (LF):"),
        dcc.Slider(id='scale', min=0.1, max=5, step=0.1, value=1.0)
    ])

# Callback para actualizar gráficos
@app.callback(
    [Output('trajectory-plot', 'figure'),
     Output('metric-plot', 'figure')],
    [Input('traj-selector', 'value'),
     Input('n-steps', 'value'),
     Input('speed', 'value'),
     Input('metric-selector', 'value')]
)
def update_plots(traj_type, n_steps, speed, metric):
    df = generate_trajectory(traj_type, n_steps, speed)
    print(f"Path Length for {traj_type}: {path_length(df)}")
    # Gráfica de trayectoria
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=df['x_pos'], y=df['y_pos'], mode='lines', name='Trayectoria'))
    fig1.update_layout(title='Trayectoria en 2D', xaxis_title='X', yaxis_title='Y')
    
    # Gráfica de métrica seleccionada
    if metric == 'PL':
        metric_value = path_length(df)
        print(f"Path Length for {traj_type}: {path_length(df)}")
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(
    x=np.arange(len(metric_value)),  # El índice como eje X (tiempo)
    y=metric_value,  
    mode='lines',
    name='Path Length'
))
        fig2.update_layout(
        title='Path Length',
        xaxis_title='Métrica',
        yaxis_title='Valor',
        bargap=0.5  # Espaciado entre barras (aquí solo hay una, pero mejora la visualización)
    )
    elif metric == 'MSD':
        metric_values = mean_squared_displacement(df)
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(y=metric_values, mode='lines', name='MSD'))
        fig2.update_layout(title='Mean Squared Displacement')
    else:
        metric_values = turning_angle_distribution(df)
         # Crear la línea en lugar del histograma
        hist, bin_edges = np.histogram(metric_values, bins=30, density=True)  # Histograma normalizado
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])  # Calcular el centro de los bins

        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(x=bin_centers, y=hist, mode='lines', name="Densidad de Ángulos", line=dict(color='blue')))

        fig2.update_layout(title='Turning Angle Distribution', xaxis_title="Ángulo (grados)", yaxis_title="Densidad")


    return fig1, fig2

# Ejecutar la aplicación
if __name__ == '__main__':
    app.run_server(debug=True)


Path Length for BM: [   5.   10.   15.   20.   25.   30.   35.   40.   45.   50.   55.   60.
   65.   70.   75.   80.   85.   90.   95.  100.  105.  110.  115.  120.
  125.  130.  135.  140.  145.  150.  155.  160.  165.  170.  175.  180.
  185.  190.  195.  200.  205.  210.  215.  220.  225.  230.  235.  240.
  245.  250.  255.  260.  265.  270.  275.  280.  285.  290.  295.  300.
  305.  310.  315.  320.  325.  330.  335.  340.  345.  350.  355.  360.
  365.  370.  375.  380.  385.  390.  395.  400.  405.  410.  415.  420.
  425.  430.  435.  440.  445.  450.  455.  460.  465.  470.  475.  480.
  485.  490.  495.  500.  505.  510.  515.  520.  525.  530.  535.  540.
  545.  550.  555.  560.  565.  570.  575.  580.  585.  590.  595.  600.
  605.  610.  615.  620.  625.  630.  635.  640.  645.  650.  655.  660.
  665.  670.  675.  680.  685.  690.  695.  700.  705.  710.  715.  720.
  725.  730.  735.  740.  745.  750.  755.  760.  765.  770.  775.  780.
  785.  790.  795.  800.  805. 

### Análisis de Longitud de Trayectoria
Se compara la longitud del camino recorrido en diferentes tipos de caminatas.

# Activity 2: Lévy Distribution - N Different Curves

In [7]:
#############################
# Mean Squared Displacement #
#############################
def msd(trajectory):
    """
    Compute the Mean Squared Displacement (MSD) for a given trajectory.
    
    Parameters:
    trajectory: A numpy array containing the trajectory
    
    Returns:
    msd: The MSD as a function of time.
    """
    N = len(trajectory)    
    msd = np.zeros(N-1)
    
    for i in range(1,N):
        displacements = trajectory[i:] - trajectory[:N - i]
        squared_displacements = np.sum(displacements**2, axis=1) # Square each coordinate of the displacement and add them together
        msd[i-1] = np.mean(squared_displacements) # Save the msd
    
    return msd

n_steps = 1000

# Definir las configuraciones de cada tipo de caminata
walks = {
    "BM_3": bm_2d(n_steps, speed=3),
    "BM_6": bm_2d(n_steps, speed=6),
    "CRW_6_c0.6": rw_2d(n_steps, speed=6, c=0.6),
    "CRW_6_c0.9": rw_2d(n_steps, speed=6, c=0.9),
    "Levy_6_alpha1": levy_flight(n_steps, alpha=1, c=0.5),
    "Levy_6_alpha0.7": levy_flight(n_steps, alpha=0.7, c=0.5)
}
# Verificar si alguna caminata devolvió None
for key, df in walks.items():
    if df is None:
        print(f"Error: La función para '{key}' devolvió None.")

# Obtener las trayectorias (solo columnas x_pos y y_pos)
trajectories = {
    key: df[['x_pos', 'y_pos']].values for key, df in walks.items() if isinstance(df, pd.DataFrame) and not df.empty
}

# Calcular MSD para cada trayectoria
msd_values = {key: msd(traj) for key, traj in trajectories.items()}

# Crear la figura
fig = go.Figure()

# Agregar trazas en un loop
for key, df in walks.items():
    if df is None:
        print(f"Advertencia: {key} es None y no se graficará.")
        continue  # Saltar esta iteración si df es None
    
    fig.add_trace(go.Scatter(
        x=df.index,
        y=msd_values.get(key, []),  # Evitar error si key no está en msd_values
        marker=dict(size=2),
        line=dict(width=2),
        mode='lines',
        name=f'MSD {key.replace("_", " ")}',
        showlegend=True
    ))

# Configuración del layout
fig.update_layout(
    title_text="Mean Squared Displacement - (BM vs CRW)",
    autosize=False,
    width=900,
    height=500,
    xaxis=dict(title="Time Step"),
    yaxis=dict(title="Mean Squared Displacement")
)

# Mostrar la gráfica
fig.show()

### Análisis de Desplazamiento Cuadrático Medio (MSD)
Se calcula el desplazamiento cuadrático medio para analizar la relación entre el tiempo y la distancia recorrida en cada tipo de caminata.

# Activity 3: Histograms + Curves


In [8]:

def trajectory_angles(trajectory):    
    angles = []

    for i in range(1, len(trajectory) - 1):
        # Get Vectors
        V1 = np.array(trajectory.iloc[i] - trajectory.iloc[i - 1]) # v(i) - v(i-1)
        V2 = np.array(trajectory.iloc[i + 1] - trajectory.iloc[i]) # v(i+1) - v(i)  
        
        # Dot Product
        dot_product = np.dot(V1, V2)
        
        # Vector magnitudes
        norm_V1 = np.linalg.norm(V1)
        norm_V2 = np.linalg.norm(V2)
        
        # Avoid div by 0
        if norm_V1 == 0 or norm_V2 == 0:
            angles.append(0)
            continue
        
        # Calculate the angle     
        cos_theta = dot_product / (norm_V1 * norm_V2)        
                
        theta = np.arccos(cos_theta)  # Cos -1  get angles on [0, 2π]

        # Convert V1 and V2 to 3D (to apply the cross product)
        V1_3D = np.array([V1[0], V1[1], 0], dtype=np.float64)
        V2_3D = np.array([V2[0], V2[1], 0], dtype=np.float64)
        
        # Calculate the cross product in 3D to determine the sign of the angle
        cross_product = np.cross(V1_3D, V2_3D)[-1]
        
        # If cross product is negative, the angle is negative
        if cross_product < 0:
            theta = -theta            
        
        angles.append(theta)  
    
    return pd.Series(angles)

cauchy1 = 0.4
cauchy2 = 0.7
n_steps = 1000
# Generate the trajectories
crw_6_c1 = rw_2d(n_steps, s_pos=[2,5], c=cauchy1)
crw_6_c2 = rw_2d(n_steps, s_pos=[2,5], c=cauchy2)

# Calculate the angles
crw_6_c1['angle'] = trajectory_angles(crw_6_c1).shift(1)  
crw_6_c2['angle'] = trajectory_angles(crw_6_c2).shift(1)

# Plot the results
resolution = 500
aux_domain = np.linspace(-np.pi, np.pi, resolution)

aux = np.mod(aux_domain, 2 * np.pi)
aux2 = np.mod(aux_domain, 2 * np.pi)

figuraWrapc_pdf = go.Figure()
wrapcauchy_1_pdf = np.array([wrapcauchy.pdf(i,cauchy1) for i in aux])
wrapcauchy_2_pdf = np.array([wrapcauchy.pdf(i,cauchy2) for i in aux2])

figuraWrapc_pdf.add_trace(go.Histogram(
    x=crw_6_c1.angle.values, 
    nbinsx=150, 
    histnorm='probability density', 
    marker=dict(color='red'),
    opacity=0.5,
    name=f'Observed Cauchy {cauchy1}'
))
figuraWrapc_pdf.add_trace(go.Histogram(
    x=crw_6_c2.angle.values, 
    nbinsx=150, 
    histnorm='probability density',
    marker=dict(color='blue'),
    opacity=0.5,
    name=f'Observed Cauchy {cauchy2}'
))
figuraWrapc_pdf.add_trace(
    go.Scatter(
        x = aux_domain,
        y = wrapcauchy_1_pdf,
        marker = dict(size=2),
        line = dict(width=2, color='red'),
        mode = 'lines',
        name=f'CRW, Cauchy {cauchy1}',
        showlegend = True
    )
)
figuraWrapc_pdf.add_trace(
    go.Scatter(
        x = aux_domain,
        y = wrapcauchy_2_pdf,
        marker = dict(size=2),
        line = dict(width=2,  color='blue'),
        mode = 'lines',            
        name=f'CRW, Cauchy {cauchy2}',
        showlegend = True
    )
)

figuraWrapc_pdf.update_layout(
    title_text = 'Turning-angle Distribution - (source dist. vs observed dist.) ',
    autosize = False,
    width=1000, 
    height=500,
    xaxis=dict(title="Turning Angle"),  # Set X-axis label
    yaxis=dict(title="Probability Density")  # Set Y-axis label
)

figuraWrapc_pdf.show()

TypeError: rw_2d() got an unexpected keyword argument 's_pos'

# Activity 4:  Step-length Distribution

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import levy_stable, wrapcauchy



# Parámetros iniciales
alpha_values = [0.9, 0.6]
m = 4.5  # Location parameter
beta = 1
n_steps = 1000
resolution = 600
x = np.linspace(0, 50, resolution)

# Generar las caminatas de Lévy y calcular las longitudes de los pasos
levy_walks = [levy_flight(n_steps, alpha=alpha, scale=1.0) for alpha in alpha_values]
step_lengths = [lw[lw <= 50] for lw in levy_walks]  # Filtrar valores ≤ 50

# Crear la figura
fig_levy_pdf = go.Figure()

# # Agregar histogramas de longitudes de paso observadas
# colors = ['blue', 'red']
# for sl, alpha, color in zip(step_lengths, alpha_values, colors):
#     fig_levy_pdf.add_trace(go.Histogram(
#         x=sl -m,
#         nbinsx=80,
#         histnorm='probability density',
#         marker=dict(color=color),
#         opacity=0.5,
#         name=f'Observed α = {alpha}'
#     ))
# Agregar histogramas de longitudes de paso observadas
colors = ['blue', 'red']
for sl, alpha, color in zip(step_lengths, alpha_values, colors):
    shift = m if alpha == 0.6 else 0  # Ajustar solo para α = 0.6
    fig_levy_pdf.add_trace(go.Histogram(
        x=sl + shift,  # Aplicar el desplazamiento adecuado
        nbinsx=80,
        histnorm='probability density',
        marker=dict(color=color),
        opacity=0.5,
        name=f'Observed α = {alpha}'
    ))
    # no pude hacer que el histograma se ajustara a la curva de la distrubucion de levi
# Agregar curvas de la distribución estable de Lévy
for alpha, color in zip(alpha_values, colors):
    fig_levy_pdf.add_trace(go.Scatter(
        x=x,
        y=levy_stable.pdf(x, alpha, beta, loc=m),
        line=dict(width=2, color=color),
        mode='lines',
        name=f'Levy α = {alpha}'
    )) 

# Configuración del gráfico
fig_levy_pdf.update_layout(
    title="Step-length Distribution - (Source vs Observed)",
    width=800, 
    height=500,
    xaxis=dict(title="Step Length", range=[0, 50]),
    yaxis=dict(title="Probability Density"),
)

fig_levy_pdf.show()
